# Replication and Extension of Brückner & Ciccone (2011)
## "Rain and the Democratic Window of Opportunity" — *Econometrica* 79(3)

This notebook replicates **Tables III–IX** of Brückner & Ciccone (2011) using the
original 1981–2004 data, then re-runs every specification on the extended 1981–2024
panel produced in `variable_construction.ipynb`.

### Estimation approach (identical to B&C)
- **OLS / 2SLS** with country fixed effects + country-specific linear time trend + year fixed effects
- **Standard errors:** Huber robust, clustered at the country level
- **Instrument:** log rainfall at *t*−1 for log GDP per capita at *t*−1 (Tables V, VII) and for the country-specific recession indicator at *t*−1 (Tables VI, VII)
- **2SLS inference:** Anderson-Rubin *p*-values (robust to weak instruments) reported in brackets

### Critical variable mapping

The original database stores `polity_change` as a *backward* difference (D_t − D_{t−1}),
while the paper's notation uses a *forward* difference ΔD_{t−1} = D_t − D_{t−1}.
This introduces a one-period offset for all regressors relative to the paper's labels:

| Paper label | `panel_final.csv` column | Notes |
|---|---|---|
| Log Rainfall, *t* | `lgpcp_l` | log rain lagged 1 yr in DB |
| Log Rainfall, *t*−1 | `lgpcp_l2` | log rain lagged 2 yrs; **the 2SLS instrument** |
| Log Rainfall, *t*−2 | `lgpcp_l3` | used in Table IV cols 2–3 |
| Log GDP, *t* (Table IV) | `lgdp_l2` | log GDP lagged 2 yrs in DB |
| Log GDP, *t*−1 (Table V endogenous) | `lgdp_l2` | same column |
| Recession, *t*−1 (Table VI endogenous) | `recession_l2` | country-specific recession indicator |
| Polity2, *t* (convergence control) | `polity2l` | Polity2 lagged 1 yr |


## Setup: Imports and Helper Functions

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore")

# ── Load panel ────────────────────────────────────────────────────────────────
panel = pd.read_csv("data/processed/panel_final.csv")
print(f"Panel: {panel.shape[0]:,} rows | "
      f"{panel['year'].min()}–{panel['year'].max()} | "
      f"{panel['iso3'].nunique()} countries")

orig = panel[panel["year"].between(1981, 2004)].copy()
full = panel.copy()
print(f"Original sample (1981–2004): {len(orig):,} rows")
print(f"Extended sample (1981–2024): {len(full):,} rows")

Panel: 1,781 rows | 1981–2024 | 41 countries
Original sample (1981–2004): 961 rows
Extended sample (1981–2024): 1,781 rows


In [2]:
# ── Econometric helpers ───────────────────────────────────────────────────────

def cluster_se(resid, X, groups):
    """Cluster-robust sandwich covariance with Stata's small-sample correction."""
    n, k    = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    scores  = X * resid[:, None]
    B = np.zeros((k, k))
    for g in np.unique(groups):
        sc = scores[groups == g].sum(axis=0)
        B += np.outer(sc, sc)
    G   = len(np.unique(groups))
    adj = (G / (G - 1)) * (n - 1) / (n - k)
    V   = adj * XtX_inv @ B @ XtX_inv
    return V, np.sqrt(np.diag(V))


def make_fe_matrix(sub):
    """Country dummies + year dummies + country×(year − mean) time trends."""
    ymean  = sub["year"].mean()
    c_dum  = pd.get_dummies(sub["iso3"], prefix="c", drop_first=True)
    y_dum  = pd.get_dummies(sub["year"], prefix="y", drop_first=True)
    ct_raw = pd.DataFrame(
        {f"ct_{c}": (sub["iso3"] == c).astype(float) * (sub["year"] - ymean)
         for c in sub["iso3"].unique()})
    ct_dum = ct_raw.iloc[:, 1:]
    return sm.add_constant(pd.concat([c_dum, y_dum, ct_dum], axis=1).astype(float))


def _partial(M_np, v):
    coef = np.linalg.lstsq(M_np, v, rcond=None)[0]
    return v - M_np @ coef


def ols_fe(df, dep, regressors, dropna_extra=None):
    """OLS with FEs + clustered SEs. Returns {coef, se, p, n}."""
    need = [dep] + regressors + (dropna_extra or [])
    sub  = df.dropna(subset=need).reset_index(drop=True)
    M_np = make_fe_matrix(sub).values
    y_r  = _partial(M_np, sub[dep].values)
    Xr   = np.column_stack([_partial(M_np, sub[r].values) for r in regressors])
    groups = sub["iso3"].values
    coef = np.linalg.lstsq(Xr, y_r, rcond=None)[0]
    _, se = cluster_se(y_r - Xr @ coef, Xr, groups)
    G = len(np.unique(groups))
    t = coef / se
    p = 2 * stats.norm.sf(np.abs(t))
    return {"coef": dict(zip(regressors, coef)),
            "se":   dict(zip(regressors, se)),
            "p":    dict(zip(regressors, p)),
            "n":    len(y_r)}


def iv2sls_fe(df, dep, endog, instrument, extra_exog=None, dropna_extra=None):
    """2SLS with FEs. Returns coef, se, AR p-value, and first-stage stats."""
    need = [dep, endog, instrument] + (extra_exog or []) + (dropna_extra or [])
    sub  = df.dropna(subset=need).reset_index(drop=True)
    M_np = make_fe_matrix(sub).values
    groups = sub["iso3"].values

    y_r = _partial(M_np, sub[dep].values)
    D_r = _partial(M_np, sub[endog].values)
    Z_r = _partial(M_np, sub[instrument].values)

    if extra_exog:
        C_r = np.column_stack([_partial(M_np, sub[c].values) for c in extra_exog])
        def pC(v):
            return v - C_r @ np.linalg.lstsq(C_r, v, rcond=None)[0]
        y_r, D_r, Z_r = pC(y_r), pC(D_r), pC(Z_r)

    Z, D = Z_r.reshape(-1, 1), D_r.reshape(-1, 1)
    fs_b  = np.linalg.lstsq(Z, D, rcond=None)[0]
    D_hat = Z @ fs_b
    b2s   = np.linalg.lstsq(D_hat, y_r, rcond=None)[0]
    _, se  = cluster_se(y_r - D.flatten() * b2s[0], D_hat, groups)

    # Anderson-Rubin (reduced form)
    rf_b   = np.linalg.lstsq(Z, y_r, rcond=None)[0]
    _, rf_se = cluster_se(y_r - Z.flatten() * rf_b[0], Z, groups)
    G = len(np.unique(groups))
    ar_p  = 2 * stats.norm.sf(abs(rf_b[0] / rf_se[0]))

    # First stage
    _, fs_se = cluster_se(D.flatten() - Z.flatten() * fs_b[0], Z, groups)
    fs_p  = 2 * stats.norm.sf(abs(fs_b[0, 0] / fs_se[0]))

    return {"coef_2s": float(b2s[0]),   "se_2s": float(se[0]),
            "ar_p":    float(ar_p),      "n":     len(y_r),
            "fs_coef": float(fs_b[0,0]), "fs_se": float(fs_se[0]),
            "fs_p":    float(fs_p)}


def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

def fmt(c, s, p, w=10):
    a = f"{c:.3f}{stars(p)}"
    return f"{a:>{w}}", f"({s:.3f})"

print("Helper functions loaded.")

Helper functions loaded.


## Table III — Rainfall and Polity Change (OLS)

**Dependent variable:** ΔPolity2 (cols 1–2) and Polity IV subscores (cols 3–5)

B&C targets (col 1): Log rainfall, *t* = 0.261 (0.347); Log rainfall, *t*−1 = −1.461\*\* (0.723)


In [3]:
regs_III = ["lgpcp_l", "lgpcp_l2"]

def table3(df_s, label):
    print(f"\n{'─'*72}")
    print(f"  {label}")
    print(f"  {'':36} (1)ΔPol2   (2)ΔPol2*  (3)ΔExc   (4)ΔPolc  (5)ΔExrec")
    print(f"  {'':38}  [all]  [no interr]")

    r1 = ols_fe(df_s, "polity_change", regs_III)
    r2 = ols_fe(df_s.dropna(subset=["exconst_change"]), "polity_change", regs_III)
    r3 = ols_fe(df_s, "exconst_change", regs_III)
    r4 = ols_fe(df_s, "polcomp_change", regs_III)
    r5 = ols_fe(df_s, "exrec_change",   regs_III)

    for reg in regs_III:
        tag = "Log rainfall, t" if reg == "lgpcp_l" else "Log rainfall, t−1"
        def f(r): return fmt(r["coef"][reg], r["se"][reg], r["p"][reg], 9)
        c1,s1=f(r1); c2,s2=f(r2); c3,s3=f(r3); c4,s4=f(r4); c5,s5=f(r5)
        print(f"  {tag:36} {c1}  {c2}  {c3}  {c4}  {c5}")
        print(f"  {'':36} {s1}  {s2}  {s3}  {s4}  {s5}")

    print(f"  {'N':36} {r1['n']:>8}  {r2['n']:>10}  {r3['n']:>7}  {r4['n']:>7}  {r5['n']:>7}")
    print(f"  * col 2 drops interregnum periods (Polity2=−66 coded as 0 in col 1)")

print("TABLE III — RAINFALL AND POLITY CHANGE")
print("OLS; country FE + country time trend + year FE; clustered SEs")
table3(orig, "Panel A: Original B&C sample (1981–2004)")
table3(full, "Panel B: Extended sample (1981–2024)")

TABLE III — RAINFALL AND POLITY CHANGE
OLS; country FE + country time trend + year FE; clustered SEs

────────────────────────────────────────────────────────────────────────
  Panel A: Original B&C sample (1981–2004)
                                       (1)ΔPol2   (2)ΔPol2*  (3)ΔExc   (4)ΔPolc  (5)ΔExrec
                                          [all]  [no interr]
  Log rainfall, t                          0.261      0.031      0.093     -0.153      0.091
                                       (0.351)  (0.386)  (0.113)  (0.154)  (0.174)
  Log rainfall, t−1                     -1.461**   -1.660**    -0.459*   -0.578**   -0.485**
                                       (0.732)  (0.750)  (0.259)  (0.290)  (0.247)
  N                                         955         902      902      902      902
  * col 2 drops interregnum periods (Polity2=−66 coded as 0 in col 1)

────────────────────────────────────────────────────────────────────────
  Panel B: Extended sample (1981–2024)
        

## Table IV — Rainfall, Per Capita GDP, and Country-Specific Recessions (OLS)

**Dependent variable:** Log real per capita GDP (cols 1–2); recession indicator (cols 5–6)

B&C targets: Log rain, *t* → Log GDP = 0.079\*\*\* (0.029); Log rain, *t* → Recession = −0.399\*\*\* (0.140)


In [4]:
def table4(df_s, label):
    print(f"\n{'─'*72}")
    print(f"  {label}")
    print(f"  {'':30} (1)logGDP  (2)logGDP+lag  (5)Reces  (6)Reces+lag")

    # Restrict to the same 955-obs base as Table III (polity_change non-missing)
    r1 = ols_fe(df_s, "lgdp_l2",     ["lgpcp_l2"],            dropna_extra=["polity_change","lgpcp_l"])
    r2 = ols_fe(df_s, "lgdp_l2",     ["lgpcp_l2","lgpcp_l3"], dropna_extra=["polity_change","lgpcp_l"])
    r5 = ols_fe(df_s, "recession_l2",["lgpcp_l2"],            dropna_extra=["polity_change","lgpcp_l"])
    r6 = ols_fe(df_s, "recession_l2",["lgpcp_l2","lgpcp_l3"], dropna_extra=["polity_change","lgpcp_l"])

    for reg, tag in [("lgpcp_l2","Log rain, t"), ("lgpcp_l3","Log rain, t−1")]:
        def fx(r):
            if reg not in r["coef"]: return " "*9, " "*7
            return fmt(r["coef"][reg], r["se"][reg], r["p"][reg], 9)
        c1,s1=fx(r1); c2,s2=fx(r2); c5,s5=fx(r5); c6,s6=fx(r6)
        print(f"  {tag:30} {c1}  {c2}        {c5}  {c6}")
        print(f"  {'':30} {s1}  {s2}        {s5}  {s6}")
    print(f"  {'N':30} {r1['n']:>8}  {r2['n']:>12}  {r5['n']:>8}  {r6['n']:>9}")

print("TABLE IV — RAINFALL, PER CAPITA GDP, AND COUNTRY-SPECIFIC RECESSIONS")
table4(orig, "Panel A: Original B&C sample (1981–2004)")
table4(full, "Panel B: Extended sample (1981–2024)")

TABLE IV — RAINFALL, PER CAPITA GDP, AND COUNTRY-SPECIFIC RECESSIONS

────────────────────────────────────────────────────────────────────────
  Panel A: Original B&C sample (1981–2004)
                                 (1)logGDP  (2)logGDP+lag  (5)Reces  (6)Reces+lag
  Log rain, t                     0.079***   0.075***        -0.399***  -0.382***
                                 (0.029)  (0.026)        (0.141)  (0.128)
  Log rain, t−1                                 0.048                      -0.191
                                          (0.033)                 (0.141)
  N                                   955           955       955        955

────────────────────────────────────────────────────────────────────────
  Panel B: Extended sample (1981–2024)
                                 (1)logGDP  (2)logGDP+lag  (5)Reces  (6)Reces+lag
  Log rain, t                        0.100      0.083         -0.203**   -0.202**
                                 (0.061)  (0.051)        (0.096)  

## Table V — Income Shocks and Polity Change (2SLS)

**Instrument:** Log rainfall, *t*−1 (`lgpcp_l2`) for Log GDP, *t*−1 (`lgdp_l2`)

B&C target (col 1): Log GDP, *t*−1 = −18.021\*\* [AR *p* ≈ 0.049]


In [5]:
def table5(df_s, label):
    print(f"\n{'─'*72}")
    print(f"  {label}")
    outcomes = [("polity_change","(1)ΔPol2"), ("exconst_change","(5)ΔExc"),
                ("polcomp_change","(6)ΔPolc"), ("exrec_change","(7)ΔExrec")]
    res = [iv2sls_fe(df_s, dep, "lgdp_l2", "lgpcp_l2") for dep,_ in outcomes]
    labels = [l for _,l in outcomes]
    print(f"  {'':26} " + "  ".join(f"{l:>12}" for l in labels))
    print(f"  {'Log GDP, t−1 (2SLS)':26} " +
          "  ".join(fmt(r['coef_2s'],r['se_2s'],r['ar_p'],12)[0] for r in res))
    print(f"  {'':26} " + "  ".join(f"({r['se_2s']:.3f}){' ':6}" for r in res))
    print(f"  {'':26} " + "  ".join(f"[AR p={r['ar_p']:.3f}]{' ':2}" for r in res))
    print(f"  {'N':26} " + "  ".join(f"{r['n']:>12}" for r in res))
    r0 = res[0]
    print(f"\n  First stage — Log GDP,t−1 ~ Log rain,t−1: "
          f"{r0['fs_coef']:.3f}{stars(r0['fs_p'])} ({r0['fs_se']:.3f})")

print("TABLE V — INCOME SHOCKS AND POLITY CHANGE (2SLS)")
table5(orig, "Panel A: Original B&C sample (1981–2004)")
table5(full, "Panel B: Extended sample (1981–2024)")

TABLE V — INCOME SHOCKS AND POLITY CHANGE (2SLS)

────────────────────────────────────────────────────────────────────────
  Panel A: Original B&C sample (1981–2004)
                                 (1)ΔPol2       (5)ΔExc      (6)ΔPolc     (7)ΔExrec
  Log GDP, t−1 (2SLS)            -18.021*       -5.809*      -7.680**       -6.137*
                             (11.488)        (3.988)        (4.731)        (4.158)      
                             [AR p=0.051]    [AR p=0.077]    [AR p=0.040]    [AR p=0.057]  
  N                                   955           902           902           902

  First stage — Log GDP,t−1 ~ Log rain,t−1: 0.079*** (0.029)

────────────────────────────────────────────────────────────────────────
  Panel B: Extended sample (1981–2024)
                                 (1)ΔPol2       (5)ΔExc      (6)ΔPolc     (7)ΔExrec
  Log GDP, t−1 (2SLS)              -5.705        -0.946        -1.724      -3.905**
                             (5.835)        (3.193)       

## Table VI — Country-Specific Recessions and Polity Change (2SLS)

**Instrument:** Log rainfall, *t*−1 (`lgpcp_l2`) for recession indicator, *t*−1 (`recession_l2`)

B&C target (col 1): Recession, *t*−1 = 3.584\*\* [AR *p* ≈ 0.049]


In [6]:
def table6(df_s, label):
    print(f"\n{'─'*72}")
    print(f"  {label}")
    df_r = df_s.dropna(subset=["recession_l2"])
    outcomes = [("polity_change","(1)ΔPol2"), ("exconst_change","(5)ΔExc"),
                ("polcomp_change","(6)ΔPolc"), ("exrec_change","(7)ΔExrec")]
    res = [iv2sls_fe(df_r, dep, "recession_l2", "lgpcp_l2") for dep,_ in outcomes]
    labels = [l for _,l in outcomes]
    print(f"  {'':26} " + "  ".join(f"{l:>12}" for l in labels))
    print(f"  {'Recession,t−1 (2SLS)':26} " +
          "  ".join(fmt(r['coef_2s'],r['se_2s'],r['ar_p'],12)[0] for r in res))
    print(f"  {'':26} " + "  ".join(f"({r['se_2s']:.3f}){' ':6}" for r in res))
    print(f"  {'':26} " + "  ".join(f"[AR p={r['ar_p']:.3f}]{' ':2}" for r in res))
    print(f"  {'N':26} " + "  ".join(f"{r['n']:>12}" for r in res))
    r0 = res[0]
    print(f"\n  First stage — Recession,t−1 ~ Log rain,t−1: "
          f"{r0['fs_coef']:.3f}{stars(r0['fs_p'])} ({r0['fs_se']:.3f})")

print("TABLE VI — COUNTRY-SPECIFIC RECESSIONS AND POLITY CHANGE (2SLS)")
table6(orig, "Panel A: Original B&C sample (1981–2004)")
table6(full, "Panel B: Extended sample (1981–2024)")

TABLE VI — COUNTRY-SPECIFIC RECESSIONS AND POLITY CHANGE (2SLS)

────────────────────────────────────────────────────────────────────────
  Panel A: Original B&C sample (1981–2004)
                                 (1)ΔPol2       (5)ΔExc      (6)ΔPolc     (7)ΔExrec
  Recession,t−1 (2SLS)             3.584*        1.130*       1.494**        1.194*
                             (1.927)        (0.632)        (0.777)        (0.680)      
                             [AR p=0.051]    [AR p=0.077]    [AR p=0.040]    [AR p=0.057]  
  N                                   955           902           902           902

  First stage — Recession,t−1 ~ Log rain,t−1: -0.399*** (0.141)

────────────────────────────────────────────────────────────────────────
  Panel B: Extended sample (1981–2024)
                                 (1)ΔPol2       (5)ΔExc      (6)ΔPolc     (7)ΔExrec
  Recession,t−1 (2SLS)              3.165        -0.243         0.329       1.459**
                             (2.273)     

## Table VII — Income Shocks, Polity Change, and Democratic Convergence

Adds the lagged Polity2 **level** as a convergence control on the right-hand side.

B&C targets: Polity2 (OLS) ≈ −0.294\*\*\* (0.023); Log rain, *t*−1 ≈ −1.404\*\* (0.690)


In [7]:
def table7(df_s, label):
    print(f"\n{'─'*72}")
    print(f"  {label}")
    r1 = ols_fe(df_s, "polity_change", ["polity2l","lgpcp_l","lgpcp_l2"])
    r3 = iv2sls_fe(df_s, "polity_change", "lgdp_l2",     "lgpcp_l2", extra_exog=["polity2l"])
    r4 = iv2sls_fe(df_s.dropna(subset=["recession_l2"]),
                   "polity_change", "recession_l2", "lgpcp_l2", extra_exog=["polity2l"])
    print(f"  {'Polity2 level (OLS)':38} {r1['coef']['polity2l']:.3f}{stars(r1['p']['polity2l'])}"
          f"  ({r1['se']['polity2l']:.3f})")
    print(f"  {'Log rain, t−1 (OLS)':38} {r1['coef']['lgpcp_l2']:.3f}{stars(r1['p']['lgpcp_l2'])}"
          f"  ({r1['se']['lgpcp_l2']:.3f})   N={r1['n']}")
    print(f"  {'Log GDP, t−1 (2SLS)':38} {r3['coef_2s']:.3f}{stars(r3['ar_p'])}"
          f"  ({r3['se_2s']:.3f})  [AR p={r3['ar_p']:.3f}]  N={r3['n']}")
    print(f"  {'Recession, t−1 (2SLS)':38} {r4['coef_2s']:.3f}{stars(r4['ar_p'])}"
          f"  ({r4['se_2s']:.3f})  [AR p={r4['ar_p']:.3f}]  N={r4['n']}")

print("TABLE VII — INCOME SHOCKS, POLITY CHANGE, AND DEMOCRATIC CONVERGENCE")
table7(orig, "Panel A: Original B&C sample (1981–2004)")
table7(full, "Panel B: Extended sample (1981–2024)")

TABLE VII — INCOME SHOCKS, POLITY CHANGE, AND DEMOCRATIC CONVERGENCE

────────────────────────────────────────────────────────────────────────
  Panel A: Original B&C sample (1981–2004)
  Polity2 level (OLS)                    -0.294***  (0.023)
  Log rain, t−1 (OLS)                    -1.401**  (0.699)   N=955
  Log GDP, t−1 (2SLS)                    -17.360**  (10.683)  [AR p=0.048]  N=955
  Recession, t−1 (2SLS)                  3.450**  (1.790)  [AR p=0.048]  N=955

────────────────────────────────────────────────────────────────────────
  Panel B: Extended sample (1981–2024)
  Polity2 level (OLS)                    -0.174***  (0.026)
  Log rain, t−1 (OLS)                    -0.710  (0.438)   N=1731
  Log GDP, t−1 (2SLS)                    -6.878  (6.777)  [AR p=0.118]  N=1731
  Recession, t−1 (2SLS)                  3.377  (2.264)  [AR p=0.125]  N=1691


## Table VIII — Rainfall and Polity Transitions (OLS)

**Dependent variables:** Binary transition indicators (OLS linear probability model)

B&C targets — Log rain, *t*−1: DemTrans = −0.125\*\* (0.057); DemStep = −0.140\*\* (0.064); AutoTrans = 0.169 (0.113); CoupDem = −0.003 (0.115)


In [8]:
regs_VIII = ["lgpcp_l", "lgpcp_l2"]

def table8(df_s, label):
    print(f"\n{'─'*72}")
    print(f"  {label}")
    print(f"  {'':36} DemTrans  DemStep  AutoTrans  CoupDem")

    dem_mask = df_s["polity2l"] > 0      # coup restricted to democracies
    r1 = ols_fe(df_s.dropna(subset=["trans_democ"]),      "trans_democ",      regs_VIII)
    r2 = ols_fe(df_s.dropna(subset=["trans_democ_epst"]), "trans_democ_epst", regs_VIII)
    r3 = ols_fe(df_s.dropna(subset=["trans_autoc"]),      "trans_autoc",      regs_VIII)
    r4 = ols_fe(df_s[dem_mask].dropna(subset=["coup"]),   "coup",             regs_VIII)

    for reg in regs_VIII:
        tag = "Log rainfall, t" if reg == "lgpcp_l" else "Log rainfall, t−1"
        def f(r): return fmt(r["coef"][reg], r["se"][reg], r["p"][reg], 9)
        c1,s1=f(r1); c2,s2=f(r2); c3,s3=f(r3); c4,s4=f(r4)
        print(f"  {tag:36} {c1}  {c2}  {c3}  {c4}")
        print(f"  {'':36} {s1}  {s2}  {s3}  {s4}")

    print(f"  {'N':36} {r1['n']:>9}  {r2['n']:>7}  {r3['n']:>9}  {r4['n']:>7}")

print("TABLE VIII — RAINFALL AND POLITY TRANSITIONS")
table8(orig, "Panel A: Original B&C sample (1981–2004)")
table8(full, "Panel B: Extended sample (1981–2024)")

TABLE VIII — RAINFALL AND POLITY TRANSITIONS

────────────────────────────────────────────────────────────────────────
  Panel A: Original B&C sample (1981–2004)
                                       DemTrans  DemStep  AutoTrans  CoupDem
  Log rainfall, t                          0.027      0.016     -0.021     -0.005
                                       (0.034)  (0.027)  (0.049)  (0.091)
  Log rainfall, t−1                     -0.125**   -0.140**      0.169     -0.003
                                       (0.058)  (0.064)  (0.115)  (0.118)
  N                                          700      867        255      255

────────────────────────────────────────────────────────────────────────
  Panel B: Extended sample (1981–2024)
                                       DemTrans  DemStep  AutoTrans  CoupDem
  Log rainfall, t                          0.033      0.016      0.009     -0.005
                                       (0.030)  (0.027)  (0.022)  (0.091)
  Log rainfall, t−1      

## Table IX — Income Shocks and Transitions to Democracy (2SLS)

B&C targets: Log GDP → DemTrans = −1.285\*\* [AR *p* = 0.027]; Recession → DemTrans = 0.235\*\* [AR *p* = 0.027]


In [9]:
def table9(df_s, label):
    print(f"\n{'─'*72}")
    print(f"  {label}")
    dem_s  = df_s.dropna(subset=["trans_democ"])
    step_s = df_s.dropna(subset=["trans_democ_epst"])
    dem_r  = dem_s.dropna(subset=["recession_l2"])
    step_r = step_s.dropna(subset=["recession_l2"])

    ls1 = ols_fe(dem_s,  "trans_democ",      ["lgdp_l2"])
    ls4 = ols_fe(step_s, "trans_democ_epst", ["lgdp_l2"])
    iv2 = iv2sls_fe(dem_s,  "trans_democ",      "lgdp_l2",     "lgpcp_l2")
    iv5 = iv2sls_fe(step_s, "trans_democ_epst", "lgdp_l2",     "lgpcp_l2")
    iv3 = iv2sls_fe(dem_r,  "trans_democ",      "recession_l2", "lgpcp_l2")
    iv6 = iv2sls_fe(step_r, "trans_democ_epst", "recession_l2", "lgpcp_l2")

    print(f"\n  {'':26} DemTrans (LS)  DemTrans (2SLS)  DemStep (LS)  DemStep (2SLS)")
    print(f"  {'Log GDP, t−1':26}"
          f"  {ls1['coef']['lgdp_l2']:.3f}{stars(ls1['p']['lgdp_l2'])} ({ls1['se']['lgdp_l2']:.3f})"
          f"    {iv2['coef_2s']:.3f}{stars(iv2['ar_p'])} ({iv2['se_2s']:.3f})"
          f"    {ls4['coef']['lgdp_l2']:.3f}{stars(ls4['p']['lgdp_l2'])} ({ls4['se']['lgdp_l2']:.3f})"
          f"    {iv5['coef_2s']:.3f}{stars(iv5['ar_p'])} ({iv5['se_2s']:.3f})")
    print(f"  {'  [AR p-value]':26}{'':14}  [{iv2['ar_p']:.3f}]"
          f"{'':12}  [{iv5['ar_p']:.3f}]")
    print(f"  {'Recession, t−1':26}"
          f"  {'—':>14}"
          f"    {iv3['coef_2s']:.3f}{stars(iv3['ar_p'])} ({iv3['se_2s']:.3f})"
          f"    {'—':>12}"
          f"    {iv6['coef_2s']:.3f}{stars(iv6['ar_p'])} ({iv6['se_2s']:.3f})")
    print(f"  {'  [AR p-value]':26}{'':14}  [{iv3['ar_p']:.3f}]"
          f"{'':12}  [{iv6['ar_p']:.3f}]")
    print(f"  N: DemTrans {ls1['n']}/{iv2['n']}/{iv3['n']}  "
          f"| DemStep {ls4['n']}/{iv5['n']}/{iv6['n']}")
    print(f"\n  First stages (instrument: lgpcp_l2):")
    print(f"    Log GDP → DemTrans:    {iv2['fs_coef']:.3f}{stars(iv2['fs_p'])} ({iv2['fs_se']:.3f})")
    print(f"    Recession → DemTrans:  {iv3['fs_coef']:.3f}{stars(iv3['fs_p'])} ({iv3['fs_se']:.3f})")

print("TABLE IX — INCOME SHOCKS AND TRANSITIONS TO DEMOCRACY (2SLS)")
table9(orig, "Panel A: Original B&C sample (1981–2004)")
table9(full, "Panel B: Extended sample (1981–2024)")

TABLE IX — INCOME SHOCKS AND TRANSITIONS TO DEMOCRACY (2SLS)

────────────────────────────────────────────────────────────────────────
  Panel A: Original B&C sample (1981–2004)

                             DemTrans (LS)  DemTrans (2SLS)  DemStep (LS)  DemStep (2SLS)
  Log GDP, t−1                0.056 (0.058)    -1.285** (0.805)    -0.054 (0.051)    -1.471** (0.820)
    [AR p-value]                            [0.029]              [0.031]
  Recession, t−1                           —    0.235** (0.133)               —    0.279** (0.126)
    [AR p-value]                            [0.029]              [0.031]
  N: DemTrans 700/700/700  | DemStep 867/867/867

  First stages (instrument: lgpcp_l2):
    Log GDP → DemTrans:    0.095** (0.037)
    Recession → DemTrans:  -0.519*** (0.166)

────────────────────────────────────────────────────────────────────────
  Panel B: Extended sample (1981–2024)

                             DemTrans (LS)  DemTrans (2SLS)  DemStep (LS)  DemStep (2SLS)
  L

## Extension Summary

How do the key results change when the sample is extended from 1981–2004 to 1981–2024?


In [10]:
rows = []
for label, df_s in [("Original (1981–2004)", orig), ("Extended (1981–2024)", full)]:
    r_ols = ols_fe(df_s, "polity_change", ["lgpcp_l","lgpcp_l2"])
    r_iv  = iv2sls_fe(df_s, "polity_change", "lgdp_l2",     "lgpcp_l2")
    r_rec = iv2sls_fe(df_s.dropna(subset=["recession_l2"]),
                      "polity_change", "recession_l2", "lgpcp_l2")
    r_dem = ols_fe(df_s.dropna(subset=["trans_democ"]), "trans_democ", ["lgpcp_l","lgpcp_l2"])
    rows.append({
        "Period":                        label,
        "OLS logRain,t−1→ΔPolity2":     f"{r_ols['coef']['lgpcp_l2']:.3f}{stars(r_ols['p']['lgpcp_l2'])}",
        "2SLS logGDP,t−1→ΔPolity2":     f"{r_iv['coef_2s']:.3f}{stars(r_iv['ar_p'])} [p={r_iv['ar_p']:.3f}]",
        "2SLS Recession→ΔPolity2":       f"{r_rec['coef_2s']:.3f}{stars(r_rec['ar_p'])} [p={r_rec['ar_p']:.3f}]",
        "OLS logRain,t−1→DemTrans":      f"{r_dem['coef']['lgpcp_l2']:.3f}{stars(r_dem['p']['lgpcp_l2'])}",
        "N (ΔPolity2)":                  r_ols["n"],
    })

sdf = pd.DataFrame(rows).set_index("Period")
print("KEY RESULTS: ORIGINAL vs. EXTENDED SAMPLE")
print(sdf.to_string())
print()
print("\nNotes:")
print("  Significance: * p<0.10  ** p<0.05  *** p<0.01")
print("  2SLS p-values are Anderson-Rubin (robust to weak instruments)")
print("  All specs: country FE + country-specific linear time trend + year FE")
print("  Clustered SEs at country level")
print()
print("  Extended democracy: Polity V (2005–2018); V-Dem v16 rescaled (2019–2024)")
print("  Extended GDP:       Penn World Tables 11.0 (2005–2023)")
print("  Extended rainfall:  GPCP v2.3 (2005–2024)")

KEY RESULTS: ORIGINAL vs. EXTENDED SAMPLE
                     OLS logRain,t−1→ΔPolity2 2SLS logGDP,t−1→ΔPolity2 2SLS Recession→ΔPolity2 OLS logRain,t−1→DemTrans  N (ΔPolity2)
Period                                                                                                                               
Original (1981–2004)                 -1.461**       -18.021* [p=0.051]        3.584* [p=0.051]                 -0.125**           955
Extended (1981–2024)                   -0.588         -5.705 [p=0.159]         3.165 [p=0.143]                -0.142***          1769


Notes:
  Significance: * p<0.10  ** p<0.05  *** p<0.01
  2SLS p-values are Anderson-Rubin (robust to weak instruments)
  All specs: country FE + country-specific linear time trend + year FE
  Clustered SEs at country level

  Extended democracy: Polity V (2005–2018); V-Dem v16 rescaled (2019–2024)
  Extended GDP:       Penn World Tables 11.0 (2005–2023)
  Extended rainfall:  GPCP v2.3 (2005–2024)


## Robustness Check: Three-Sample Comparison

To assess whether the weakening of results in the extended sample reflects
**data quality** (V-Dem rescaling noise in 2019–2024) or a genuine
**structural change** in the rainfall–income–democracy mechanism, we re-run
the key specifications on three samples:

| Sample | Democracy source | N (ΔPolity2) |
|---|---|---|
| **Original** (1981–2004) | Polity IV (B&C) | 955 |
| **Polity-only** (1981–2018) | Polity IV + Polity V — no V-Dem | ~1,523 |
| **Full** (1981–2024) | Polity IV + Polity V + V-Dem rescaled | ~1,769 |

If V-Dem rescaling were the culprit, results should hold for 1981–2018 but
break only in 1981–2024. If instead results weaken already at 1981–2018,
the cause is structural — specifically, the weakening of the
rainfall → GDP transmission channel after 2004.


In [11]:
# ── Three-sample robustness check ────────────────────────────────────────────

orig = panel[panel["year"].between(1981, 2004)].copy()   # original B&C
mid  = panel[panel["year"].between(1981, 2018)].copy()   # Polity-only (no V-Dem)
full = panel.copy()                                       # full extended

samples = [
    ("Original (1981–2004)",  orig),
    ("Polity-only (1981–2018)", mid),
    ("Full (1981–2024)",      full),
]

def compare(title, rows):
    print(f"\n{'─'*72}")
    print(f"  {title}")
    print(f"  {'Sample':26} {'Coef':>10}  {'SE':>8}  {'Sig':>4}  N")
    for lbl, coef, se, p, n in rows:
        print(f"  {lbl:26} {coef:>10.3f}  ({se:.3f})  {stars(p):<4}  {n}")

# ── Table III: Log rain, t-1 → ΔPolity2 (OLS) ───────────────────────────────
rows = []
for lbl, df_s in samples:
    r = ols_fe(df_s, "polity_change", ["lgpcp_l", "lgpcp_l2"])
    c, s, p = r["coef"]["lgpcp_l2"], r["se"]["lgpcp_l2"], r["p"]["lgpcp_l2"]
    rows.append((lbl, c, s, p, r["n"]))
compare("Table III — Log rainfall, t−1 → ΔPolity2 (OLS)", rows)

# ── Table IV: Log rain, t → Log GDP (instrument strength) ───────────────────
rows = []
for lbl, df_s in samples:
    r = ols_fe(df_s, "lgdp_l2", ["lgpcp_l2"],
               dropna_extra=["polity_change", "lgpcp_l"])
    c, s, p = r["coef"]["lgpcp_l2"], r["se"]["lgpcp_l2"], r["p"]["lgpcp_l2"]
    rows.append((lbl, c, s, p, r["n"]))
compare("Table IV — Log rainfall, t → Log GDP (first-stage strength)", rows)

# ── Table V: Log GDP → ΔPolity2 (2SLS) ──────────────────────────────────────
print(f"\n{'─'*72}")
print("  Table V — Log GDP, t−1 → ΔPolity2 (2SLS; instrument: lgpcp_l2)")
print(f"  {'Sample':26} {'2SLS coef':>10}  {'AR p-val':>10}  {'FS coef':>8}  {'FS sig':>6}  N")
for lbl, df_s in samples:
    r = iv2sls_fe(df_s, "polity_change", "lgdp_l2", "lgpcp_l2")
    print(f"  {lbl:26} {r['coef_2s']:>10.3f}  [{r['ar_p']:.3f}]{stars(r['ar_p']):<4}"
          f"  {r['fs_coef']:>8.3f}{stars(r['fs_p'])}  {'':4}  {r['n']}")

# ── Table VI: Recession → ΔPolity2 (2SLS) ───────────────────────────────────
print(f"\n{'─'*72}")
print("  Table VI — Recession, t−1 → ΔPolity2 (2SLS; instrument: lgpcp_l2)")
print(f"  {'Sample':26} {'2SLS coef':>10}  {'AR p-val':>10}  {'FS coef':>8}  {'FS sig':>6}  N")
for lbl, df_s in samples:
    df_r = df_s.dropna(subset=["recession_l2"])
    r = iv2sls_fe(df_r, "polity_change", "recession_l2", "lgpcp_l2")
    print(f"  {lbl:26} {r['coef_2s']:>10.3f}  [{r['ar_p']:.3f}]{stars(r['ar_p']):<4}"
          f"  {r['fs_coef']:>8.3f}{stars(r['fs_p'])}  {'':4}  {r['n']}")

# ── Table VIII: Log rain, t-1 → DemTrans (OLS) ──────────────────────────────
rows = []
for lbl, df_s in samples:
    r = ols_fe(df_s.dropna(subset=["trans_democ"]), "trans_democ",
               ["lgpcp_l", "lgpcp_l2"])
    c, s, p = r["coef"]["lgpcp_l2"], r["se"]["lgpcp_l2"], r["p"]["lgpcp_l2"]
    rows.append((lbl, c, s, p, r["n"]))
compare("Table VIII — Log rainfall, t−1 → Democratic Transition (OLS)", rows)

# ── Table IX: Log GDP → DemTrans (2SLS) ─────────────────────────────────────
print(f"\n{'─'*72}")
print("  Table IX — Log GDP, t−1 → Democratic Transition (2SLS)")
print(f"  {'Sample':26} {'2SLS coef':>10}  {'AR p-val':>10}  {'FS coef':>8}  {'FS sig':>6}  N")
for lbl, df_s in samples:
    dem_s = df_s.dropna(subset=["trans_democ"])
    r = iv2sls_fe(dem_s, "trans_democ", "lgdp_l2", "lgpcp_l2")
    print(f"  {lbl:26} {r['coef_2s']:>10.3f}  [{r['ar_p']:.3f}]{stars(r['ar_p']):<4}"
          f"  {r['fs_coef']:>8.3f}{stars(r['fs_p'])}  {'':4}  {r['n']}")

# ── Interpretation ───────────────────────────────────────────────────────────
print(f"\n{'─'*72}")
print("  INTERPRETATION")
print(f"  {'─'*68}")
print("""
  Rainfall → ΔPolity2 (Table III): already insignificant at 1981–2018,
  before any V-Dem data enters. V-Dem rescaling is NOT the cause.

  First-stage strength (Table IV): rainfall → GDP weakens from 0.079***
  (original) to 0.097* (1981–2018) to 0.100 n.s. (1981–2024). The
  instrument loses power in the post-2004 period, which drives the
  attenuation of all 2SLS results (Tables V–VI).

  Democratic transitions (Tables VIII–IX): strengthen monotonically
  across all three samples, unaffected by data source. The binary
  transition mechanism is robust to both structural change and data splice.
""")


────────────────────────────────────────────────────────────────────────
  Table III — Log rainfall, t−1 → ΔPolity2 (OLS)
  Sample                           Coef        SE   Sig  N
  Original (1981–2004)           -1.461  (0.732)  **    955
  Polity-only (1981–2018)        -0.725  (0.474)        1523
  Full (1981–2024)               -0.588  (0.404)        1769

────────────────────────────────────────────────────────────────────────
  Table IV — Log rainfall, t → Log GDP (first-stage strength)
  Sample                           Coef        SE   Sig  N
  Original (1981–2004)            0.079  (0.029)  ***   955
  Polity-only (1981–2018)         0.097  (0.059)  *     1523
  Full (1981–2024)                0.100  (0.061)        1769

────────────────────────────────────────────────────────────────────────
  Table V — Log GDP, t−1 → ΔPolity2 (2SLS; instrument: lgpcp_l2)
  Sample                      2SLS coef    AR p-val   FS coef  FS sig  N
  Original (1981–2004)          -18.021  [0.051